# 19 | Value-RNN paradigm A: TD-trained RNNs on a Pavlovian contingency task

This notebook trains a recurrent network with **temporal-difference (TD) learning**,
the learning rule behind the reward-prediction-error theory of dopamine, and
reproduces the RNN modeling of

> Qian, Burrell, Hennig, Matias, Murthy, Gershman & Uchida (2025), *Prospective contingency explains behavior and dopamine signals during associative learning*, Nature Neuroscience.

**The paper's question.** Midbrain dopamine neurons behave like the TD error of
reinforcement learning, $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$. But a TD
model is only as good as its state representation: what is "the state" at each
moment of an experiment? The paper attacks this question with a Pavlovian
*contingency degradation* task in mice. Odor A predicts reward with probability
0.75 and odor B predicts nothing. When extra rewards are then delivered *uncued*
during the inter-trial interval (ITI), anticipatory licking and the dopamine
response to odor A collapse, even though the odor A -> reward contingency itself
has not changed. When the same extra rewards are *signaled* by a third odor C,
the odor A response survives. Classical contingency measures ($\Delta P$) cannot
separate these two conditions; a TD model can, but only if its state
representation captures the animal's uncertainty about where it is within the
ITI. The paper's best hand-crafted model, the Belief-State model, represents the
ITI as a probability distribution over a "Wait" and a "Pre" state. Its final
question is the one this notebook is about: instead of hand-crafting states,
can a network *learn* an appropriate state representation from observations
alone?

**The value-RNN.** A small GRU with a scalar linear readout $V_t$ is trained with
TD(0) to predict reward, given only the same cue and reward observations the
animal receives. After the same two-phase training as the animals (conditioning
first, then degradation or cued reward), the network's TD errors reproduce the
dopamine results in all conditions, its pre-cue value explains *why* (uncued
rewards make waiting itself valuable), and its hidden units linearly decode the
hand-crafted beliefs, the better the larger the network. Trained with the TD
error as its only feedback, the RNN discovers a belief-like state space on its
own.

**How this maps onto the NeuralRNN layers.** Notebook 18 used PPO: an online
environment, an actor-critic agent, a dedicated `RLTrainer`. TD value learning
is simpler. It is a *training objective*, so it plugs into the ordinary
supervised pipeline, and the environment is replaced by an offline episodic
dataset:

| layer | notebook 18 (PPO actor-critic) | this notebook (value-RNN) |
|---|---|---|
| data | `rl.envs`, stepped online | `ContingencyDataset` (`"contingency"`): offline episodes of 20 concatenated trials |
| model | `ActorCriticModel` (policy + value heads) | any RNN core with a scalar readout (here `gated_rnn`, GRU) |
| loss | `PPOLoss` + `RLTrainer` | `TDObjective` (TD(0) semi-gradient) + the generic `Trainer` |
| analysis | behavior + population geometry | `rl.probe` (trial labeling, cue-aligned averaging, RPE at events) + notebook-local belief-state utils |

**Contents**

1. Setup
2. The contingency task (`ContingencyDataset`)
3. The TD objective and the value-RNN
4. Two-phase training: conditioning, then degradation or cue-c (Fig. 6b)
5. Asymptotic RPE predictions per condition (Fig. 6c)
6. Example value and TD-error traces (Fig. 6d)
7. CCA-aligned state-space trajectories (Fig. 6e/g)
8. Decoding the hand-crafted beliefs (Fig. 6f)
9. Summary

**Scale.** The paper trains 25 networks per condition on 10,000-trial sessions
(300 epochs per phase). Here we train 2 seeds on 2,000-trial sessions for fewer
steps, which runs on a CPU in tens of minutes; every pattern pointed out below
is robust to this reduction, and the text says where the paper's numbers
differ. All stages are load-first: checkpoints under `models/19/` are reused
when present, so delete them to retrain.

## 1. Setup

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from neuralrnn import (
    AutoModel, TDObjective, Trainer, TrainingArguments, load_dataset,
)
from neuralrnn.models.gated_rnn import GatedRNNConfig
from neuralrnn.rl.probe import (
    probe_value_model, split_episode_trials, label_trial_type,
    average_by_aligned_time, mean_rpe_at_events, shift_iti,
)

MODEL_DIR = "models/19"
FIG_DIR = "figs/19"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# ---- paper hyperparameters (Qian & Burrell 2024, Methods) ----
GAMMA = 0.83          # TD discount at dt = 0.5 s (0.67 per second)
ISI = 8               # cue -> reward delay in steps (rew_times = [8, 8, 8])
LR = 1e-3             # Adam
BATCH_SIZE = 12       # episodes per batch
TRIALS_PER_EPISODE = 20

# ---- reduced tutorial config ----
N_TRIALS = 2000                  # paper: 10000
STEPS_CONDITIONING = 3000        # paper: 300 epochs ~= 12.5k steps
STEPS_TRANSFER = 2000
SEEDS = [0, 1]                   # paper: 25 simulations
HIDDEN_MAIN = 50                 # Fig. 6c/d/e/g
HIDDEN_SWEEP = [5, 10, 20, 50]   # Fig. 6f
MODES = ("conditioning", "degradation", "cue-c")


def make_dataset(mode, seed, ntrials=N_TRIALS, jitter=1):
    """Training dataset for one mode (reward timing jittered, as the paper)."""
    return load_dataset("contingency", mode=mode, ntrials=ntrials,
                        ntrials_per_episode=TRIALS_PER_EPISODE, jitter=jitter,
                        batch_size=BATCH_SIZE, seed=seed)


def make_test_dataset(mode, seed=1000, ntrials=1000):
    """Jitter-free dataset for display/analysis (paper removes jitter for display)."""
    return make_dataset(mode, seed=seed, ntrials=ntrials, jitter=0)


def new_model(hidden):
    """Value-RNN: GRU + scalar linear readout (PyTorch default init, as the paper)."""
    return AutoModel.from_config(GatedRNNConfig(
        input_dim=4, latent_dim=hidden, output_dim=1, rnn_type="GRU"))


def ckpt_path(tag):
    return os.path.join(MODEL_DIR, tag)

## 2. The contingency task

`ContingencyDataset` (registered as `"contingency"`) simulates the Pavlovian
sessions of the experiment. A session is a stream of trials, and one trial is

$$\underbrace{\text{nothing}}_{\text{ITI}} \;\longrightarrow\;
\text{odor onset} \;\longrightarrow\;
\underbrace{\text{nothing}}_{\text{ISI}} \;\longrightarrow\;
\text{reward or omission},$$

discretized at dt = 0.5 s. The ISI (odor onset to outcome) is fixed at 8 steps
(4 s); the ITI is a truncated geometric draw (p = 1/12, kept within [8, 20]
steps) on top of a 20-step minimum, so it lasts 28-40 steps (14-20 s) and its
end cannot be anticipated from timing alone. This unpredictability is the
point of the experiment: how the model represents the eventless ITI decides
whether it can explain degradation at all.

The model observes four channels: one-hot odor A, B, C and a reward channel
(which is also the TD target). Trial types repeat in proportions 2/5 odor A,
1/5 odor B, 2/5 "blank", block-shuffled in blocks of 20 trials. The three
conditions differ only in what fills the blank slots:

| condition | odor A | odor B | blank slots | mouse behavior |
|---|---|---|---|---|
| conditioning | reward, p = 0.75 | never rewarded | nothing happens | strong anticipatory licking to A |
| degradation | reward, p = 0.75 | never rewarded | **uncued** reward, p = 0.75 | licking to A collapses |
| cue-c (cued reward) | reward, p = 0.75 | never rewarded | odor C -> reward, p = 0.75 | licking to A preserved |

Degradation and cue-c deliver exactly the same rewards; only their
predictability differs. That is the paper's control against satiety and
reward-rate accounts of contingency.

Two properties of the dataset matter for everything below:

* **Episodes, not trials, are the unit of training.** Twenty consecutive
  trials are concatenated into one stream and the RNN state is carried across
  the whole stream, so the ITI between two trials is experienced exactly like
  the eventless periods inside a trial. `sample_batch()` returns a batch of
  12 episodes, zero-padded with a mask.
* **Trial labels come from the inputs, not the simulator state** (the paper's
  analysis convention). A degradation trial shows no cue, so input-based
  labeling calls it "Degradation"; an unrewarded odor-A trial looks like an
  odor-B trial until the outcome time has passed. The printed counts below
  show the resulting label distribution: in conditioning, 310 of 400 odor-A
  trials are rewarded (~ 0.75), and the 400 blank slots of one condition
  become 400 degradation or cue-C trials in the others.

Training data jitter the reward time by one step in either direction (the
paper's way of adding internal timing uncertainty); test and display data are
jitter-free. The plot shows one example trial per cue/outcome for each
condition; consecutive examples are laid out along the time axis.

In [ ]:
# One example trial per cue/outcome for each condition
CH_STYLE = [("cue A", "tab:green"), ("cue B", "tab:gray"),
            ("cue C", "tab:blue"), ("reward", "tab:red")]
fig, axes = plt.subplots(3, 1, figsize=(9, 6.5))
for ax, mode in zip(axes, MODES):
    ds = make_test_dataset(mode)
    shown, t_off = set(), 0
    for trial in ds.trials:
        key = (trial["cue"], trial["rewarded"])
        if key in shown:
            continue
        shown.add(key)
        for ch, (name, color) in enumerate(CH_STYLE):
            ts = np.where(trial["X"][:, ch] > 0)[0] + t_off
            ax.eventplot([ts], lineoffsets=ch, linelengths=0.8, colors=color)
        t_off += trial["length"] + 5
        if len(shown) >= 5:
            break
    ax.set_yticks(range(4), [name for name, _ in CH_STYLE])
    ax.set_title(f'condition = "{mode}"   reward probs [A, B, C] = '
                 f'{ds.rew_probs}   cue C shown: {ds.cue_shown[2]}',
                 fontsize=10)
    ax.set_xlim(-2, t_off)
axes[-1].set_xlabel("time (steps of 0.5 s); consecutive example trials offset along x")
fig.suptitle("The contingency task: example trials per condition")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "task_demo.png"), dpi=150)
plt.show()

# Trial-type statistics (labeled from the *inputs*, so hidden-cue trials
# appear as Degradation / Blank: the analysis convention of the paper)
from collections import Counter
for mode in MODES:
    ds = make_test_dataset(mode)
    print(f"{mode:13s}", dict(Counter(label_trial_type(t["X"]) for t in ds.trials)))

## 3. The TD objective and the value-RNN

The value-RNN is a *critic-only* agent: there is no policy, because the task
is Pavlovian and the animal's only job is to predict what comes next. The
model is a plain GRU whose hidden state
$\mathbf{z}_t = f_\phi(\mathbf{o}_t, \mathbf{z}_{t-1})$ plays the role of
the state representation, with a scalar linear readout of value,

$$V_t = \mathbf{w}^\top \mathbf{z}_t + w_0.$$

All parameters (the GRU's $\phi$ and the readout $\mathbf{w}, w_0$) are
trained end-to-end with the semi-gradient TD(0) loss

$$\delta_t = r_{t+1} + \gamma\, V_{t+1}\big|_{\text{stop-grad}} - V_t,
\qquad \mathcal{L} = \mathbb{E}_t\big[\delta_t^2\big],$$

implemented as `TDObjective` (registered as `"td"`). Three details of the
loss are worth knowing:

* *Semi-gradient*: the bootstrap target $r_{t+1} + \gamma V_{t+1}$ is treated
  as a constant (`.detach()`), exactly as in the paper; gradients flow only
  through $V_t$.
* *Reward indexing*: $r_{t+1}$ is the reward channel of the dataset's target
  stream, so the TD error at the step *before* an observed reward is what
  corresponds to the dopamine response at that reward.
* *Masking*: a TD error is counted only when both $t$ and $t+1$ are valid
  steps, so padded episode tails never leak into the bootstrap.

The discount factor is $\gamma = 0.83$ per 0.5 s step, a discount rate of
0.67 per second, chosen by the paper to match the $\gamma = 0.925$ of its
0.2 s tabular TD simulations. Because `TDObjective` is an ordinary objective,
the generic `Trainer` drives the optimization (Adam, learning rate $10^{-3}$,
batches of 12 episodes), and `keep_best=True` restores the lowest-loss
weights at the end, matching the reference's argmin-over-epochs checkpoint
selection.

The cell below sanity-checks the objective on one batch: an untrained network
already has a small loss (rewards are sparse, so predicting near zero is not
a bad start) and a mean predicted value near zero.

In [ ]:
demo_model = new_model(HIDDEN_MAIN)
demo_obj = TDObjective(gamma=GAMMA)
demo_ds = make_dataset("conditioning", seed=0)
loss0, logs = demo_obj.compute_loss(demo_model, demo_ds.sample_batch())
print(f"untrained TD loss: {loss0.item():.4f}  logs: {logs}")

## 4. Two-phase training: conditioning, then degradation or cue-c

The networks go through the same protocol as the mice (Fig. 6b): every
network is first trained to asymptote on *conditioning*, and the main models
are then *retrained* on either degradation or cue-c. This transfer design is
what lets section 7 compare conditions within one state space: the
degradation and cue-c models differ from the conditioning model only by their
second-phase experience.

Two groups of networks are trained:

| group | phases | hidden units | seeds | used in |
|---|---|---|---|---|
| main | conditioning -> degradation, conditioning -> cue-c | 50 | 2 | sections 5-7 (Fig. 6c/d/e/g) |
| conditioning-only | conditioning | 5, 10, 20, 50 | 1 | section 8 (Fig. 6f sweep) |

Paper hyperparameters kept: Adam at learning rate $10^{-3}$, batches of 12
episodes of 20 trials, jittered reward timing during training. Reduced for
the tutorial: 2,000-trial sessions (paper: 10,000), 3,000 + 2,000 gradient
steps for the two phases (paper: 300 epochs, ~12,500 steps), 2 seeds (paper:
25). Everything is load-first: existing checkpoints under `models/19/` are
reused; delete them to retrain (a full reduced run takes roughly 30-45 min on
CPU).

In [ ]:
def train_one(model, dataset, steps, tag, seed=0):
    args = TrainingArguments(max_steps=steps, learning_rate=LR,
                             batch_size=BATCH_SIZE, keep_best=True,
                             device="cpu", seed=seed, log_every=100,
                             output_dir=ckpt_path(tag))
    trainer = Trainer(model, dataset, TDObjective(gamma=GAMMA), args)
    trainer.train()
    model.save_pretrained(ckpt_path(tag))  # keep_best weights already restored
    return trainer


def train_conditioning_then(transfer_mode, hidden=HIDDEN_MAIN, seed=0):
    cond_tag = f"cond_h{hidden}_s{seed}"
    xfer_tag = f"{transfer_mode}_h{hidden}_s{seed}"
    if os.path.exists(os.path.join(ckpt_path(xfer_tag), "config.json")):
        return AutoModel.from_pretrained(ckpt_path(xfer_tag))
    if os.path.exists(os.path.join(ckpt_path(cond_tag), "config.json")):
        model = AutoModel.from_pretrained(ckpt_path(cond_tag))
    else:
        torch.manual_seed(seed); np.random.seed(seed)
        model = new_model(hidden)
        train_one(model, make_dataset("conditioning", seed),
                  STEPS_CONDITIONING, cond_tag, seed)
    train_one(model, make_dataset(transfer_mode, seed + 100),
              STEPS_TRANSFER, xfer_tag, seed)
    return model


TRAIN = True   # set False to only load existing checkpoints
if TRAIN:
    for seed in SEEDS:
        for mode in ("degradation", "cue-c"):
            train_conditioning_then(mode, hidden=HIDDEN_MAIN, seed=seed)
            print(f"done: {mode} h{HIDDEN_MAIN} s{seed}")
    for hidden in HIDDEN_SWEEP:
        tag = f"cond_h{hidden}_s{SEEDS[0]}"
        if not os.path.exists(os.path.join(ckpt_path(tag), "config.json")):
            torch.manual_seed(SEEDS[0]); np.random.seed(SEEDS[0])
            train_one(new_model(hidden), make_dataset("conditioning", SEEDS[0]),
                      STEPS_CONDITIONING, tag, SEEDS[0])
            print(f"done: {tag}")

The training curve (mean squared TD error per logged step) drops quickly at
first, as the readout learns the average reward rate of the session, and then
keeps decaying slowly while the recurrent dynamics refine the cue and ITI
representations needed for precise timing. The mean predicted value settles
at the average discounted reward rate of the conditioning session.

In [ ]:
# Training curve of one conditioning run (logged by the generic Trainer)
import json
hist_path = os.path.join(ckpt_path(f"cond_h{HIDDEN_MAIN}_s{SEEDS[0]}"),
                         "history.json")
hist = json.load(open(hist_path))
steps = [h["step"] for h in hist]
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(steps, [h["loss"] for h in hist], color="tab:blue")
ax.set_xlabel("gradient step")
ax.set_ylabel("TD loss (mean $\delta^2$)", color="tab:blue")
ax.tick_params(axis="y", labelcolor="tab:blue")
if "mean_value" in hist[0]:
    ax2 = ax.twinx()
    ax2.plot(steps, [h["mean_value"] for h in hist], color="tab:orange",
             alpha=0.8)
    ax2.set_ylabel("mean predicted value $V$", color="tab:orange")
    ax2.tick_params(axis="y", labelcolor="tab:orange")
ax.set_title(f"conditioning phase (h = {HIDDEN_MAIN}, seed {SEEDS[0]})")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "training_curve.png"), dpi=150)
plt.show()

## 5. Asymptotic RPE predictions per condition (Fig. 6c)

The paper's central comparison: the TD error at each task event, read out
from a trained network, should match the dopamine response recorded at that
event in the corresponding condition. We probe each trained model on fresh
jitter-free data of *its own* condition (the conditioning model on
conditioning, and so on), label trials from the inputs, and average the RPE
at seven events: odor A onset, the rewarded and omitted outcomes of odor A,
odor B onset, odor C onset and outcome, and the uncued background reward of
degradation.

Two analysis conventions of the paper are built into `rl.probe` and matter
for reading the bars:

* *Cue-aligned time.* Each trial is re-aligned so its first event (cue onset,
  or the hidden-cue time for degradation trials) sits at $t = 0$; degradation
  trials are shifted by ISI + 1 so that the *unpredicted* background reward
  lands one step after where a predicted reward would have been, which is how
  the paper overlays conditions.
* *RPE indexed by transition start.* The RPE of the transition $t \to t+1$
  is plotted at $t$, so the odor RPE appears at $-1$ and the reward RPE at
  ISI $- 1$ relative to outcome time.

The expected pattern (Fig. 6c of the paper, and the mouse dopamine data of
its Fig. 2): the odor A RPE is large in conditioning, **collapses in
degradation**, and is **restored in cue-c**; the RPE at the odor-A reward is
positive (the reward is only 75% predictable, so a sizeable surprise
remains) and nearly identical across conditions; the omission RPE is
negative; and the uncued background reward in degradation evokes a large
positive RPE. Bars show the mean over seeds with s.d. error bars (paper: n =
25 networks; here 2 seeds, so treat the error bars as indicative only). The
numeric values are printed under the figure.

In [ ]:
MODEL_FOR_MODE = {"conditioning": "cond", "degradation": "degradation",
                  "cue-c": "cue-c"}
EVENTS = [("Cue A Rewarded", "cue", "odor A onset", "tab:green"),
          ("Cue A Rewarded", "reward", "reward after A", "tab:red"),
          ("Cue A Unrewarded", "reward", "omission after A", "tab:brown"),
          ("Cue B", "cue", "odor B onset", "tab:gray"),
          ("Cue C Rewarded", "cue", "odor C onset", "tab:blue"),
          ("Cue C Rewarded", "reward", "reward after C", "tab:pink"),
          ("Degradation", "reward", "uncued reward", "tab:purple")]


def event_rpes(mode, seed, n_episodes=10):
    tag = f"{MODEL_FOR_MODE[mode]}_h{HIDDEN_MAIN}_s{seed}"
    model = AutoModel.from_pretrained(ckpt_path(tag))
    ds = make_test_dataset(mode)
    records = probe_value_model(model, ds, n_episodes=n_episodes, gamma=GAMMA)
    trials = [t for r in records for t in split_episode_trials(r)]
    return mean_rpe_at_events(trials, isi=ISI)


# collect per-seed event RPEs for every condition
table = {}
for mode in MODES:
    per_seed = [event_rpes(mode, s) for s in SEEDS]
    table[mode] = {}
    for tt, ev, label, color in EVENTS:
        vals = [d[tt][ev] for d in per_seed if tt in d and np.isfinite(d[tt][ev])]
        if vals:
            table[mode][label] = (float(np.mean(vals)),
                                  float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2), sharey=True)
for ax, mode in zip(axes, MODES):
    labels = [l for _, _, l, _ in EVENTS if l in table[mode]]
    means = [table[mode][l][0] for l in labels]
    sds = [table[mode][l][1] for l in labels]
    colors = [c for _, _, l, c in EVENTS if l in table[mode]]
    x = np.arange(len(labels))
    ax.bar(x, means, yerr=sds, color=colors, alpha=0.85, capsize=3)
    for xi, m in zip(x, means):
        ax.text(xi, m + 0.02 * np.sign(m if m else 1), f"{m:.2f}",
                ha="center", va="bottom" if m >= 0 else "top", fontsize=8)
    ax.axhline(0, color="k", lw=0.5)
    ax.set_xticks(x, labels, rotation=30, ha="right", fontsize=9)
    ax.set_title(f"{mode} model (n = {len(SEEDS)} seeds)", fontsize=11)
axes[0].set_ylabel("mean RPE at event")
fig.suptitle("Fig. 6c: asymptotic value-RNN RPE predictions per condition")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig6c_rpe_bars.png"), dpi=150)
plt.show()

print(f"{'event':<18s}" + "".join(f"{m:>16s}" for m in MODES))
for _, _, label, _ in EVENTS:
    row = f"{label:<18s}"
    for mode in MODES:
        row += f"{table[mode][label][0]:>16.3f}" if label in table[mode]                else f"{'--':>16s}"
    print(row)

The bars recover the paper's signature. The odor A RPE, large in the
conditioning model, drops sharply in the degradation model and returns in the
cue-c model. The reward after odor A evokes the same RPE
in all three conditions (about 0.35; the residual surprise of a 75%-predicted
reward, matching the dopamine data of the paper's Extended Data Fig. 6),
omissions are negative, and the uncued background reward is the single
largest event the degradation model ever sees. Note also odor B: its RPE,
near zero in conditioning, turns negative in degradation, because odor B
resets the network from the valuable Pre period back to the Wait period,
again as in the paper's Fig. 5. Because odor A's own contingency is identical in all three
conditions, the difference in its RPE can only come from the context the
network has built around it, which is precisely the paper's argument.

## 6. Example value and TD-error traces (Fig. 6d)

Why does the odor A RPE collapse in degradation? TD errors are *differences*:
at odor onset, $\delta = \gamma V(\text{ISI}) - V(\text{pre-cue})$. The
degradation manipulation could lower $\delta$ in two ways: by lowering the
ISI value (odor A predicts less than before) or by raising the pre-cue value
(the ITI itself has become valuable, because uncued rewards arrive there).
The paper resolves this by plotting the value trace itself: the ISI value is
essentially unchanged across conditions, while the **ITI value rises** in
degradation. Waiting has become valuable, so the transition into the odor-A
ISI is no longer a positive surprise.

The panels show trial-averaged value and TD error aligned to odor A onset
(dotted vertical line) with the outcome time marked (dashed line), for the
conditioning model on conditioning data (left) and the degradation model on
degradation data (right). The red curves are the uncued-reward trials, which
only the degradation model ever experiences.

In [ ]:
def aligned_curves(tag, mode, n_episodes=10):
    model = AutoModel.from_pretrained(ckpt_path(tag))
    records = probe_value_model(model, make_test_dataset(mode),
                                n_episodes=n_episodes, gamma=GAMMA)
    trials = [t for r in records for t in split_episode_trials(r)]
    return {key: average_by_aligned_time(trials, key=key) for key in ("V", "rpe")}


seed = SEEDS[0]
panels = [(f"cond_h{HIDDEN_MAIN}_s{seed}", "conditioning"),
          (f"degradation_h{HIDDEN_MAIN}_s{seed}", "degradation")]
TYPES = [("Cue A Rewarded", "tab:green", "odor A, rewarded"),
         ("Cue A Unrewarded", "tab:olive", "odor A, unrewarded"),
         ("Degradation", "tab:red", "uncued reward")]
fig, axes = plt.subplots(2, 2, figsize=(11, 6.5), sharex=True)
for col, (tag, mode) in enumerate(panels):
    curves = aligned_curves(tag, mode)
    for tt, color, label in TYPES:
        for row, key in enumerate(("V", "rpe")):
            if tt not in curves[key]:
                continue
            t, m, sem, n = curves[key][tt]
            axes[row, col].plot(t, m, color=color,
                                label=f"{label} (n = {n})" if row == 0 else None)
            axes[row, col].fill_between(t, m - sem, m + sem, color=color,
                                        alpha=0.2)
    for row in (0, 1):
        axes[row, col].axvline(0, color="k", ls=":", lw=0.8)
        axes[row, col].axvline(ISI, color="k", ls="--", lw=0.8)
    axes[0, col].set_title(f"{mode} model", fontsize=11)
    axes[0, col].legend(fontsize=8)
    axes[1, col].set_xlabel("aligned time (steps of 0.5 s; 0 = odor onset, 8 = outcome)")
axes[0, 0].set_ylabel("value $V_t$")
axes[1, 0].set_ylabel("TD error $\delta_t$")
fig.suptitle("Fig. 6d: trial-averaged value and TD error")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig6d_value_rpe_traces.png"), dpi=150)
plt.show()

Read the green value curves just *before* time 0: the pre-cue value of the
degradation model sits visibly higher than that of the conditioning model,
while the value reached inside the ISI is similar in both. The odor A RPE
(the green bump at time 0 in the lower panels) shrinks accordingly. Note also
the red curves: an uncued reward produces a large positive RPE at outcome
time and immediately resets the value trace, exactly the event that teaches
the network that the ITI is worth waiting in.

## 7. CCA-aligned state-space trajectories (Fig. 6e/g)

The value-RNN was never told about the Belief-State model's states. Do its
hidden dynamics nonetheless organize the task in the same way? Comparing
state spaces *across conditions* needs a common coordinate system, because
the degradation- and cue-c-trained networks are different models with
different hidden bases. The paper's solution: take one trained network, probe
it on all three conditions, z-score and PCA-reduce the hidden activity of
each condition separately (keeping 95% of variance), and align the
per-condition PCA spaces with multi-view **canonical correlation analysis**
(`fit_cca` in `neuralrnn.analysis`). CCA finds, for each view, a linear
projection that is maximally correlated with the corresponding projections
of the other views, so trajectories from different conditions can be drawn in
one shared 3-D space (the first three canonical variates).

Three patterns of the paper are worth looking for:

* the **odor A ISI trajectory overlaps across conditions**: the cue-evoked
  dynamics are shared, because the structure of an odor-A trial never
  changes;
* in the cue-c model, the **odor C trajectory tracks the odor A trajectory**:
  the network generalizes between the two reward-predicting cues;
* the **ITI return trajectories of the degradation model are distinct**:
  after an uncued reward the state takes a long, separate path back towards
  the ITI region, the signature of the heterogeneous, Wait-state-like ITI
  representation that the Belief-State model has by construction.

Each panel is one trained network (left: degradation-trained, right:
cue-c-trained), probed on all three conditions. Colors mark the trial type;
solid lines are the model's own training condition, dashed lines the two
transfer conditions.

In [ ]:
from matplotlib.lines import Line2D
from neuralrnn.analysis import fit_cca

PLOT_TYPES = [("Cue A Rewarded", "tab:green", "odor A, rewarded"),
              ("Cue B", "tab:gray", "odor B"),
              ("Cue C Rewarded", "tab:blue", "odor C, rewarded"),
              ("Degradation", "tab:red", "uncued reward")]


def mean_trajectory(trials, trial_type, T_max=40, pre_cue_steps=4):
    bucket = {}
    for trial in trials:
        if trial["trial_type"] != trial_type:
            continue
        aligned = shift_iti(trial["aligned_time"], pre_cue_steps)
        for a, z in zip(aligned, trial["Z"]):
            if -pre_cue_steps <= a < T_max:
                bucket.setdefault(int(a), []).append(z)
    if not bucket:
        return None, None
    times = np.array(sorted(bucket))
    traj = np.stack([np.mean(bucket[t], axis=0) for t in times])
    return times, traj


fig = plt.figure(figsize=(14, 6))
for col, trained_mode in enumerate(("degradation", "cue-c")):
    model = AutoModel.from_pretrained(
        ckpt_path(f"{trained_mode}_h{HIDDEN_MAIN}_s{SEEDS[0]}"))
    per_mode = {}
    for mode in MODES:
        records = probe_value_model(model, make_test_dataset(mode),
                                    n_episodes=8, gamma=GAMMA)
        per_mode[mode] = [t for r in records for t in split_episode_trials(r)]
    acts = [np.concatenate([t["Z"] for t in per_mode[m]], axis=0) for m in MODES]
    space = fit_cca(acts, var_threshold=0.95, n_components=3)

    ax = fig.add_subplot(1, 2, col + 1, projection="3d")
    for view, mode in enumerate(MODES):
        for tt, color, label in PLOT_TYPES:
            times, traj = mean_trajectory(per_mode[mode], tt)
            if traj is None:
                continue
            V = space.transform(traj, view)
            own = mode == trained_mode
            ax.plot(V[:, 0], V[:, 1], V[:, 2], ls="-" if own else "--",
                    color=color, alpha=0.95 if own else 0.55, lw=1.6)
    ax.set_title(f"{trained_mode}-trained RNN", fontsize=11)
    ax.set_xlabel("CC1"); ax.set_ylabel("CC2"); ax.set_zlabel("CC3")

handles = [Line2D([], [], color=c, lw=2, label=l) for _, c, l in PLOT_TYPES]
handles += [Line2D([], [], color="k", ls="-", lw=2, label="training condition"),
            Line2D([], [], color="k", ls="--", lw=2, label="transfer condition")]
fig.legend(handles=handles, fontsize=9, loc="center right",
           bbox_to_anchor=(1.0, 0.5))
fig.suptitle("Fig. 6e/g: hidden-state trajectories in the CCA-aligned space")
fig.tight_layout(rect=(0, 0, 0.86, 1))
fig.savefig(os.path.join(FIG_DIR, "fig6eg_cca_trajectories.png"), dpi=150)
plt.show()

In the cue-c-trained network the blue (odor C) trajectory rides on top of the
green (odor A) one: having learned that both cues predict the same outcome,
the network routes them through the same ISI dynamics. In the
degradation-trained network the red (uncued reward) trajectory separates from
the rest and rejoins the ITI region only slowly: the state space has become
heterogeneous where the Belief-State model places its Wait-to-Pre continuum.
The odor A trajectories themselves (green, solid vs dashed) are similar
across conditions, as in the paper.

## 8. Decoding the hand-crafted beliefs (Fig. 6f)

The strongest version of the paper's claim: the value-RNN's state space
resembles the Belief-State model's, and more so the larger the network. The
Belief-State model is a hidden Markov model over 10 latent task states: 8 ISI
substates (one per 0.5 s step from odor onset to outcome), a **Wait** state
(early ITI, right after an outcome) and a **Pre** state (late ITI, when the
next trial has become likely). Its transition matrix
$T_{ij} = P(s_{t+1} = j \mid s_t = i)$ and observation matrix
$O_{ij,e} = P(\text{event } e \mid s_t \to s_{t+1})$ over the five events
(null, reward, odor A, B, C) are estimated empirically from a simulated
session with latent-state labels (`simulate_belief_experiment`,
`estimate_transition`, `estimate_observation`). Given an observed event
stream, the belief trajectory is the HMM forward filter

$$b_t \;=\; \mathrm{normalize}\Big( b_{t-1}\,
\big(T \odot O_{:,:,\,e_t}\big) \Big),$$

a 10-dimensional probability vector at every step. To ask whether the RNN
encodes these beliefs, a linear map from hidden activity to beliefs is fit by
least squares, $B \approx [Z, \mathbf{1}]\,W$, and scored with

$$R^2 = 1 - \frac{\mathrm{var}(B - \hat B)}{\mathrm{var}(B)},$$

computed over all 10 beliefs together and for the Wait state alone
(`belief_regression_r2`). Each conditioning-only model of the hidden-size
sweep is probed on fresh conditioning data, and the beliefs are computed from
the same observed inputs the network saw.

The paper's result (its Fig. 6f): belief R² **grows with the number of hidden
units**, and the improvement concentrates in the Wait state, the ITI
representation whose heterogeneity separates degradation from the other
conditions.

### Belief-state reference model (notebook utilities)

The functions below are the Belief-State TD reference model of the paper,
ported from the authors' MATLAB code. They are specific to this Pavlovian
paradigm, so they live in this notebook rather than in the package. The port
keeps the reference's conventions exactly (1-based event codes; the belief
relabeling that collapses the post-outcome ISI of unrewarded trials into the
Wait state; the minimum-norm least-squares decoder), so the belief R² values
are directly comparable to the paper's.

In [ ]:
"""# Inlined from the reference project port (formerly neuralrnn.analysis.belief_state).
# Belief-state reference model and belief decoding (value-RNN analysis).

Ports the Belief-State TD model used in Qian & Burrell (2024) as the
handcrafted-state reference against which value-RNN hidden activity is
compared (their Fig. 3 and Fig. 6f). Source:
``matlab_simulation_code/useful_functions/{simulate_experiment_rnn,
simulate_trial, calculate_transition, calculate_observation, belief_r2}.m``
and ``matlab_simulation_code/td_belief.m``.

Components:

* :func:`simulate_belief_experiment` — generate a (states, events) stream for
  one condition ("conditioning" / "degradation" / "cue-c" / "extinction")
  with latent state labels (ISI substates, Wait, Pre).
* :func:`estimate_transition` / :func:`estimate_observation` — empirical
  transition matrix T and observation matrix O from a labeled simulation.
* :func:`belief_filter` — HMM forward filter giving the belief trajectory
  b_t over latent states for an observed event stream.
* :func:`belief_regression_r2` — linear decode of beliefs from hidden-unit
  activity Z; returns total R^2 and per-belief R^2 (the "belief R^2" of
  value-RNN analyses; the wait-state readout is column index 8).

Conventions (reference, dt = 0.5 s): events are 1 = null, 2 = reward,
3 = odor A, 4 = odor B, 5 = odor C. Latent states before belief relabeling:
ISI substates 0..8 (cue at 0, reward at 8) and the ITI wait chain 8..20
(state 8 shared). The reference's belief relabeling (applied to every trial)
maps all states in (7, 20) -> 19 ("Wait"), keeping 20 ("Pre"), and shifts
unrewarded cued trials by +7 (capped at 20) so their ISI sits in Wait; after
dense renumbering the 10 distinct states are 0..7 (ISI), 8 (Wait), 9 (Pre).
These are the b1..b10 beliefs decoded in the reference analyses.
"""
from __future__ import annotations

import numpy as np

CONDITIONS = ("conditioning", "degradation", "cue-c", "extinction")

# Event ids (match the reference's 1-based event codes)
EV_NULL, EV_REWARD, EV_A, EV_B, EV_C = 1, 2, 3, 4, 5
N_EVENTS = 5


def simulate_belief_experiment(n_trials: int, condition: str, dt: float = 0.5,
                               p_reward: float = 0.75, seed: int | None = None):
    """Simulate one experiment with latent belief-state labels.

    Returns:
        states: (T,) int array of latent state ids (0-based, relabeled dense).
        events: (T,) int array of event ids (EV_*).
    """
    if condition not in CONDITIONS:
        raise ValueError(f"condition must be one of {CONDITIONS}, got {condition!r}")
    rng = np.random.RandomState(seed)

    # --- state layout (reference: cue 1s + reward delay 2.5s = ISI 3.5s) ---
    isi_len = 1.0 + 2.5
    isi_states = np.arange(0, int(isi_len / dt) + 2)          # 0..8 at dt=0.5
    max_isi = isi_states[-1]
    # events on ISI states: first = cue placeholder (-1), last = reward
    # placeholder (0), middle = null
    isi_events = np.ones(len(isi_states), dtype=int)
    isi_events[0], isi_events[-1] = -1, 0

    post_reward, pre_reward = 4.0, 2.0
    wait = post_reward + pre_reward
    itiwait_states = np.arange(max_isi, max_isi + int(wait / dt) + 1)  # 8..20
    itiwait_events = np.ones(len(itiwait_states), dtype=int)
    iti_final = itiwait_states[-1]

    trial_dist = {
        "conditioning": ["CSp", "CSp", "CSm", "blank", "blank"],
        "degradation": ["CSp", "CSp", "CSm", "degrade", "degrade"],
        "cue-c": ["CSp", "CSp", "CSm", "CSp2", "CSp2"],
        "extinction": ["CSpExtinct", "CSpExtinct", "CSm", "blank", "blank"],
    }[condition]
    sequence = rng.choice(len(trial_dist), size=n_trials)
    trial_types = [trial_dist[i] for i in sequence]

    states_all, events_all = [], []
    for tt in trial_types:
        s, e = _simulate_trial(tt, rng, dt, p_reward, isi_states, isi_events,
                               itiwait_states, itiwait_events, iti_final)
        states_all.append(s)
        events_all.append(e)
    states = np.concatenate(states_all)
    events = np.concatenate(events_all)

    # Relabel states to dense 0..S-1 in sorted order (reference does this).
    uniq = np.unique(states)
    remap = {u: i for i, u in enumerate(uniq)}
    states = np.array([remap[s] for s in states], dtype=int)
    return states, events


def _simulate_trial(trial_type, rng, dt, p_reward, isi_states, isi_events,
                    itiwait_states, itiwait_events, iti_final):
    """One trial (cue, ISI, reward/omission) + trailing ITI, with latent states."""
    # Trailing ITI: truncated exponential (mean 6 s, keep [4, 10] s), minus 4 s.
    iti_len = rng.exponential(6.0)
    while iti_len < 4 or iti_len > 10:
        iti_len = rng.exponential(6.0)
    iti_len -= 4.0
    n_iti = int(np.ceil(iti_len / dt))
    itis = np.full(n_iti, iti_final, dtype=int)
    itie = np.ones(n_iti, dtype=int)

    pre_states = np.full(2, iti_final, dtype=int)
    pre_events = np.ones(2, dtype=int)

    p_rand = rng.rand()
    if trial_type in ("CSp", "CSp2", "CSpExtinct"):
        states = np.concatenate([pre_states, isi_states, itiwait_states, itis])
        events = np.concatenate([pre_events, isi_events, itiwait_events, itie])
        events[events == -1] = {"CSp": EV_A, "CSp2": EV_C, "CSpExtinct": EV_A}[trial_type]
        rewarded = trial_type != "CSpExtinct" and p_rand < p_reward
        events[events == 0] = EV_REWARD if rewarded else EV_NULL
    elif trial_type == "CSm":
        states = np.concatenate([pre_states, itiwait_states, itis,
                                 np.full(len(isi_states), iti_final, dtype=int)])
        events = np.concatenate([pre_events, isi_events, itiwait_events, itie])
        events[events == -1] = EV_B
        events[events == 0] = EV_NULL
    elif trial_type == "blank":
        n = 2 + len(isi_events) + len(itiwait_events) + n_iti
        states = np.full(n, iti_final, dtype=int)
        events = np.ones(n, dtype=int)
    elif trial_type == "degrade":
        states = np.concatenate([pre_states, np.full(len(isi_states), iti_final),
                                 itiwait_states, itis])
        events = np.concatenate([pre_events, isi_events, itiwait_events, itie])
        events[events == -1] = EV_NULL        # no cue shown
        events[events == 0] = EV_REWARD if p_rand < p_reward else EV_NULL
        events = np.concatenate([[EV_NULL], events[:-1]])   # align reward
    else:
        raise ValueError(f"Unknown trial type {trial_type!r}")

    # Belief labeling: on unrewarded cued trials the true state during the ISI
    # is the Wait state (no reward will come); collapse ISI substates.
    max_isi_m1 = isi_states[-1] - 1
    if trial_type in ("CSp", "CSp2") and not (p_rand < p_reward):
        states = np.minimum(states + max_isi_m1, iti_final)
    mask = (states > max_isi_m1) & (states < iti_final)
    states[mask] = iti_final - 1
    return states, events


def estimate_transition(states: np.ndarray, S: int | None = None) -> np.ndarray:
    """Empirical transition matrix T[i, j] = P(s_{t+1}=j | s_t=i)."""
    if S is None:
        S = int(states.max()) + 1
    counts = np.zeros((S, S))
    out_counts = np.zeros(S)
    for a, b in zip(states[:-1], states[1:]):
        counts[a, b] += 1
        out_counts[a] += 1
    with np.errstate(divide="ignore", invalid="ignore"):
        T = np.where(out_counts[:, None] > 0, counts / np.maximum(out_counts[:, None], 1), 0.0)
    return T


def estimate_observation(events: np.ndarray, states: np.ndarray, S: int | None = None,
                         n_events: int = N_EVENTS) -> np.ndarray:
    """Empirical observation matrix O[i, j, e] = P(event e | transition i->j).

    Events are 1-based (EV_*); index e in the last axis is 0-based
    (event id e+1). The event is the one emitted *at the arrival step*
    (events[t+1] for the transition states[t] -> states[t+1]).
    """
    if S is None:
        S = int(states.max()) + 1
    counts = np.zeros((S, S, n_events))
    for a, b, e in zip(states[:-1], states[1:], events[1:]):
        counts[a, b, e - 1] += 1
    totals = counts.sum(axis=2, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        O = np.where(totals > 0, counts / np.maximum(totals, 1), 0.0)
    return O


def belief_filter(events: np.ndarray, T: np.ndarray, O: np.ndarray,
                  b0: np.ndarray | None = None) -> np.ndarray:
    """HMM forward filter over latent states (port of td_belief's belief part).

    b_t = normalize( b_{t-1} @ (T * O[:, :, e_t]) )   (row-vector convention)

    Args:
        events: (T,) event ids (1..N_EVENTS).
        T: (S, S) transition matrix.
        O: (S, S, n_events) observation matrix.
        b0: initial belief (default: uniform).

    Returns:
        (T, S) belief trajectory; out[t] is the belief at step t BEFORE
        observing event e_t (i.e. the posterior over events up to t-1),
        matching the reference's ``results.b(t) = b0`` convention.
    """
    S = T.shape[0]
    b = np.ones(S) / S if b0 is None else np.asarray(b0, dtype=float)
    out = np.zeros((len(events), S))
    for t, e in enumerate(events):
        out[t] = b
        b = b @ (T * O[:, :, e - 1])
        s = b.sum()
        if s > 0 and np.isfinite(s):
            b = b / s
        else:
            # degenerate belief (zero-probability observation): reset to Pre
            b = np.zeros(S)
            b[-1] = 1.0
    return out


def belief_regression_r2(Z: np.ndarray, B: np.ndarray):
    """Decode beliefs from hidden-unit activity; total and per-belief R^2.

    Fits B ~= [Z, 1] @ W by least squares (minimum-norm solution, as the
    reference's lsqminnorm), then

        R^2 = 1 - ||B - B_est||_F^2 / ||B - colmean(B)||_F^2

    (the reference centers both numerator and denominator columns).

    Args:
        Z: (T, H) hidden-unit activity.
        B: (T, K) belief trajectories.

    Returns:
        (r2_total, r2_per_belief (K,), W)
    """
    Z = np.asarray(Z, dtype=float)
    B = np.asarray(B, dtype=float)
    Zb = np.hstack([Z, np.ones((Z.shape[0], 1))])
    W = np.linalg.lstsq(Zb, B, rcond=None)[0]
    B_est = Zb @ W
    numer = B - B_est
    numer = numer - numer.mean(axis=0)
    denom = B - B.mean(axis=0)
    var_n = np.sum(numer ** 2)
    var_d = np.sum(denom ** 2)
    r2_total = 1.0 - var_n / var_d
    # per-belief R^2 (reference r2_matrix: uncentered numerator, centered denom)
    ssr = np.sum((B - B_est) ** 2, axis=0)
    sst = np.sum((B - B.mean(axis=0)) ** 2, axis=0)
    with np.errstate(divide="ignore", invalid="ignore"):
        r2_per = np.where(sst > 0, 1.0 - ssr / np.maximum(sst, 1e-12), np.nan)
    return r2_total, r2_per, W

In [ ]:
N_BELIEF_STATES = 10   # b1..b10 of the reference (dense states 0..9)
WAIT_COL = 8           # Wait-state belief column


def events_from_X(X):
    """Input channels -> belief-model events (1=null, 2=reward, 3=A, 4=B, 5=C)."""
    e = np.ones(X.shape[0], dtype=int)
    e[X[:, 3] > 0] = 2
    e[X[:, 0] > 0] = 3
    e[X[:, 1] > 0] = 4
    e[X[:, 2] > 0] = 5
    return e


def belief_r2_for_model(tag, n_episodes=10):
    model = AutoModel.from_pretrained(ckpt_path(tag))
    records = probe_value_model(model, make_test_dataset("conditioning"),
                                n_episodes=n_episodes, gamma=GAMMA)
    states, events_sim = simulate_belief_experiment(2000, "conditioning", seed=0)
    T = estimate_transition(states)
    O = estimate_observation(events_sim, states)
    r2s, r2_waits = [], []
    for r in records:
        B = belief_filter(events_from_X(r["X"]), T, O)[:, :N_BELIEF_STATES]
        r2_total, r2_per, _ = belief_regression_r2(r["Z"], B)
        r2s.append(r2_total)
        r2_waits.append(r2_per[WAIT_COL])
    return float(np.mean(r2s)), float(np.mean(r2_waits))


totals, waits = [], []
for hidden in HIDDEN_SWEEP:
    r2, r2_wait = belief_r2_for_model(f"cond_h{hidden}_s{SEEDS[0]}")
    totals.append(r2); waits.append(r2_wait)
    print(f"h = {hidden:3d}:  belief R2 = {r2:.3f}   Wait-state R2 = {r2_wait:.3f}")

fig, ax = plt.subplots(figsize=(5.5, 4))
ax.plot(HIDDEN_SWEEP, totals, "o-", color="tab:blue", label="all 10 beliefs")
ax.plot(HIDDEN_SWEEP, waits, "s--", color="tab:orange",
        label="Wait state only")
for h, t, w in zip(HIDDEN_SWEEP, totals, waits):
    ax.annotate(f"{t:.2f}", (h, t), textcoords="offset points",
                xytext=(6, 6), fontsize=8, color="tab:blue")
ax.set_xlabel("number of hidden units")
ax.set_ylabel("belief $R^2$ (linear decode from hidden activity)")
ax.set_xticks(HIDDEN_SWEEP)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=9)
ax.set_title("Fig. 6f: larger value-RNNs are more belief-like")
fig.tight_layout()
fig.savefig(os.path.join(FIG_DIR, "fig6f_belief_r2.png"), dpi=150)
plt.show()

The sweep reproduces the paper's trend: the total belief R² climbs from about
0.3 at 5 units to about 0.98 at 50 units, and the Wait-state R² climbs with
it. A 50-unit GRU trained only to predict reward has learned an almost
complete linear embedding of the hand-crafted beliefs, while a 5-unit network
cannot afford one. The paper reads this as evidence that TD learning alone,
given enough capacity, discovers belief-like state representations from raw
observations, and points to the Wait-state decoding as the signature of the
heterogeneous ITI representation that degradation requires.

## 9. Summary

* TD learning is an *objective*, not a trainer. `TDObjective` plus the
  generic `Trainer` turns any RNN core with a scalar readout into a
  value-RNN; no dedicated RL machinery is needed for critic-only paradigms
  (contrast with the PPO actor-critic of notebook 18).
* `ContingencyDataset` ports the paper's task: block-shuffled cues (2/5 A,
  1/5 B, 2/5 blank), truncated-geometric ITIs, jittered reward timing, and
  episodes of 20 concatenated trials with the RNN state carried across
  trials.
* `rl.probe` ports the paper's analysis conventions (input-based trial
  labels, cue-aligned time with the degradation shift, RPE indexed by
  transition start), so value and RPE analyses transfer directly from the
  paper.
* The results reproduce Fig. 6 of the paper at reduced scale: the odor A RPE
  collapses in degradation and is restored in cue-c because the *ITI* value
  rises (sections 5-6); the state space shares the odor-A ISI trajectory
  across conditions but develops distinct ITI return paths in the degradation
  model (section 7); and larger networks linearly decode more of the
  hand-crafted beliefs (section 8).

For the full-scale protocol (10,000-trial sessions, 300 epochs per phase, 25
simulations, hidden sizes 5-50), increase `N_TRIALS`, `STEPS_*` and `SEEDS`
in the setup cell.